# 🚀 Phase 2: From Data to Decision (Strategic UX Components)

In this phase, we transform the Machine Learning models from "black-box" predictors into a strategic decision-making engine for Managers. We implement three core components designed for the frontend:

1. The **"Driver Radar" (Feature Importance)**
Purpose: To move beyond simply stating "there is risk" and instead identify what is driving that risk within the organization.

- Visualization: A dynamic analysis of the model's coefficients, presented as a feature importance bar chart.

- Key Insight: For example, the model might reveal that "90% of the risk is not driven by workload, but by the perceived stigma of speaking with a supervisor."

- Value: This provides the strategic justification for why the GenAI Marketing Campaign should prioritize a specific theme (e.g., psychological safety) over another (e.g., time management).

2. The **"Persona Deep Dive" (Cluster Profiles)**
Purpose: To humanize the data and give segments a clear identity, allowing the Manager to visualize exactly who they are supporting.

- Visualization: Descriptive profile cards for each cluster with creative names such as "The Silent Strivers" or "The Supported Peers."

- Content: * Traits: Behavioral and demographic patterns discovered during clustering.

Risk Level: Criticality scale (Low / Medium / High).

Actionable: Tactical suggestions (e.g., "This group requires short, direct Telegram messages emphasizing anonymity").

- Value: It transforms "Cluster 0" into a relatable human segment with specific needs.

3. The **"Strategic What-If" Simulator (Impact Analysis)**
Purpose: The highest level of decision support. An interactive panel where the Manager acts as a strategist.

- The Dynamics: A "sandbox" where the user can manipulate sliders for key variables (e.g., "What happens if we improve benefits awareness by 20%?").

- The Logic: The frontend instantly recalculates the wellbeing probability using the weights from our Logistic Regression model.

- Value: It allows the Manager to simulate future scenarios and predict the ROI of wellness interventions before any budget is spent.

### 💡 Technical Integration Note
To ensure a seamless user experience, the model's coefficients and cluster profiles are exported into a lightweight JSON format. This enables the frontend to execute the prediction logic locally and instantly, providing a reactive and "live" feel to the simulation without additional backend calls.

In [3]:
import joblib

# 1. Load the model
model = joblib.load('../data/models/best_model.joblib')

# 2. Extract the "weights" for the What-If Simulator and the Radar
intercept = model.intercept_[0]
coefficients = dict(zip(model.feature_names_in_, model.coef_[0]))

print("--- COEFFICIENTS AND WEIGHTS ---")
print(f"Intercept: {intercept}")
print(f"Coefficients: {coefficients}")

# 3. Example of a 'person' looks like in theJSON file.
import json
with open('../data/for_genai/persona_profiles.json', 'r') as f:
    personas = json.load(f)
    print("\n--- PEOPLE STRUCTURE ---")
    print(json.dumps(personas[0] if isinstance(personas, list) else personas, indent=2))

# 4. Example of the content of feature_importance
with open('../data/for_genai/feature_importance.json', 'r') as f:
    feat_imp = json.load(f)
    print("\n--- FEATURE IMPORTANCE ---")
    print(feat_imp)

--- COEFFICIENTS AND WEIGHTS ---
Intercept: -2.68678323641554
Coefficients: {'past_disorder': np.float64(-0.3011952618986645), 'diagnosed_flag': np.float64(0.10308045477193813), 'productivity_impact': np.float64(2.1387633766641683), 'family_history': np.float64(0.4619018816675531), 'knows_benefits': np.float64(-0.11050780144019591), 'prev_employer_support': np.float64(0.11202975962630367), 'fear_consequences': np.float64(-2.7640350637554314), 'leave_easiness': np.float64(0.1652624320156121), 'support_index': np.float64(1.6018800320016064), 'stigma_index': np.float64(-1.3723034405948138)}

--- PEOPLE STRUCTURE ---
{
  "Protected Advocates": {
    "traits": "High supervisor trust, high benefit awareness, low perceived stigma.",
    "visual_keywords": "Open communication, supportive environment, bright, collaborative.",
    "risk_level": "Low"
  },
  "Silent Risk Group": {
    "traits": "High stigma index, high fear of consequences, history of mental health disorders but low disclosure.",

## 1. Driver Radar (Feature Importance)

// For the bar chart, the highlight of the factors that most influence employee comfort (the Target).
// Suggested JSON for the chart:
````
[
  { "feature": "Productivity Impact", "value": 2.13, "impact": "Positive", "desc": "Awareness of how mental health affects work improves comfort." },
  { "feature": "Support Index", "value": 1.60, "impact": "Positive", "desc": "Higher perceived support correlates with higher supervisor trust." },
  { "feature": "Fear of Consequences", "value": -2.76, "impact": "Negative", "desc": "Main barrier: Fear of negative career impact strongly reduces comfort." },
  { "feature": "Stigma Index", "value": -1.37, "impact": "Negative", "desc": "Perceived corporate stigma prevents open conversations." },
  { "feature": "Family History", "value": 0.46, "impact": "Positive", "desc": "Familiarity with mental health often leads to seeking more support." }
]
````

## 2. The "What-If" Simulator (Mathematical Logic)

````
const predictRisk = (inputs) => {
    const intercept = -2.68678;
    const weights = {
        past_disorder: -0.3012,
        diagnosed_flag: 0.1031,
        productivity_impact: 2.1388,
        family_history: 0.4619,
        knows_benefits: -0.1105,
        prev_employer_support: 0.112,
        fear_consequences: -2.764,
        leave_easiness: 0.1653,
        support_index: 1.6019,
        stigma_index: -1.3723
    };

    // the Z value
    let z = intercept;
    for (let key in weights) {
        z += weights[key] * (inputs[key] || 0);
    }

    // the Sigmoid function to obtain the probability (0 to 1)
    const probability = 1 / (1 + Math.exp(-z));
    return (probability * 100).toFixed(1); // Returns the "Comfort" %
};
````

## 3. The "Synthetic Clients" for the Demo


In [11]:
import json

model_assets = {
    "intercept": -2.68678323641554,
    "weights": {
        'past_disorder': -0.301195,
        'diagnosed_flag': 0.103080,
        'productivity_impact': 2.138763,
        'family_history': 0.461901,
        'knows_benefits': -0.110507,
        'prev_employer_support': 0.112029,
        'fear_consequences': -2.764035,
        'leave_easiness': 0.165262,
        'support_index': 1.601880,
        'stigma_index': -1.372303
    },
    "syntheticProfiles": [
        {
            "id": "profile_01",
            "name": "The Guarded Professional",
            "description": "High performance but high fear of career impact. Reluctant to disclose due to perceived corporate stigma.",
            "inputs": {
                "past_disorder": 1, "diagnosed_flag": 0, "productivity_impact": 0.8,
                "family_history": 0, "knows_benefits": 0.2, "prev_employer_support": 0.1,
                "fear_consequences": 1, "leave_easiness": 0.2, "support_index": 0.1, "stigma_index": 0.9
            },
            "expected_status": "Critical Risk"
        },
        {
            "id": "profile_02",
            "name": "The Empowered Ally",
            "description": "Strong trust in leadership and high awareness of company benefits. Feels safe discussing mental health.",
            "inputs": {
                "past_disorder": 0, "diagnosed_flag": 0, "productivity_impact": 0.2,
                "family_history": 1, "knows_benefits": 1, "prev_employer_support": 0.8,
                "fear_consequences": 0, "leave_easiness": 0.9, "support_index": 0.9, "stigma_index": 0.1
            },
            "expected_status": "High Well-being"
        },
        {
            "id": "profile_03",
            "name": "The Uncertain Contributor",
            "description": "The perfect 'What-If' case. Moderate stigma and low benefit awareness.",
            "inputs": {
                "past_disorder": 0, "diagnosed_flag": 0, "productivity_impact": 0.5,
                "family_history": 0, "knows_benefits": 0.3, "prev_employer_support": 0.4,
                "fear_consequences": 0.5, "leave_easiness": 0.5, "support_index": 0.4, "stigma_index": 0.5
            },
            "expected_status": "Moderate Risk"
        }
    ]
}

# Generate the JavaScript string in the format 'export const'
js_content = f"""
export const modelWeights = {json.dumps(model_assets['weights'], indent=2)};
export const modelIntercept = {model_assets['intercept']};
export const syntheticProfiles = {json.dumps(model_assets['syntheticProfiles'], indent=2)};
"""

file_path = "../frontend/src/constants/modelData.js"

try:
    with open(file_path, "w") as f:
        f.write(js_content)
    print(f"✅ File successfully generated in: {file_path}")
except FileNotFoundError:
    with open("modelData.js", "w") as f:
        f.write(js_content)
    print("⚠️ I couldn't find the frontend folder. I saved 'modelData.js' in your current folder. Move it to src/constants/")

✅ File successfully generated in: ../frontend/src/constants/modelData.js


### Synthetic Persona Profiles for Demo

| Persona | Key Drivers | Business Context | Expected UI Output |
| :--- | :--- | :--- | :--- |
| **The Guarded Professional** | Max Fear & Stigma | Senior talent afraid that seeking help will stall their promotion. | 🔴 **Red / Critical** |
| **The Empowered Ally** | Max Support & Benefits | Employee who leverages company resources to maintain a healthy work-life balance. | 🟢 **Green / Healthy** |
| **The Uncertain Contributor** | Mid-range Sliders | Needs a push. Increasing "Knows Benefits" by 20% moves them from Yellow to Green. | 🟡 **Yellow / Warning** |